# Agents, MCP & Skills

En praktisk workshop hvor vi udforsker den nye verden af **agentic software engineering**.

---

## Agenda (ca. 3,5 timer)

| Tid | Emne | Øvelser |
|-----|------|---------|
| 20 min | Intro, Recap & Jupyter Notebooks med Kotlin | |
| 70 min | Agenter - teori og byg din egen | **Øvelse 1:** Tool Execute-funktioner, **Øvelse 2:** Human-in-the-loop |
| 20 min | *Pause* | |
| 45 min | Abstraktionsstigen – MCP | **Øvelse 3:** Higher-level Tools |
| 20 min | Extension: get_nearest_station | **Øvelse 4:** Nyt tool |
| 25 min | Skills – teori og prøv med GitHub Copilot | SKILL.md |
| 10 min | Afrunding | |

---

# Del 0: Intro, Recap & Jupyter Notebooks med Kotlin
*20 minutter*

#### ℹ️ Installér Ollama nu, da det tager lidt tid

**Windows (PowerShell):**
```powershell
winget install Ollama.Ollama
```

**Mac (Homebrew):**
```bash
brew install ollama
```

Start Ollama og hiv modellen ned:
```bash
ollama serve
ollama pull llama3.1:8b
```

👆🏻 kan også foretages i GUI; vælg model, skriv en prompt (f.eks. "hej") og tryk retur

## Opsummering fra sidste gang

- LLMs & Agents - What are they, and why can they code? — Martin
- Agentisk softwareudvikling for personlig produktivitet — Emil
- Agentisk Software Engineering for fuld SDLC af et produkt — Magnus

Lige denne planche er meget relevant mht. dagens seance: [How LLMs Learn: The Three Stages](https://martinrl.github.io/presentations/what-are-llms-and-agents/index.html#14)

<img src="images/LLMs-14.png" alt="LLMs-14" width="600" style="max-width: 100%;">

---

## Intro til Jupyter Notebooks med Kotlin

### Hvad er en Jupyter Notebook med Kotlin?

- `.ipynb` format (JSON-baseret)
- Kører på Kotlin Jupyter kernel (kotlin-jupyter)
- Understøtter Kotlin med fuld JVM-adgang
- Har adgang til Maven/Gradle dependencies via `@file:DependsOn`

### Sådan kører du en celle

- **Shift+Enter** - Kør celle og gå til næste
- **Ctrl+Enter** - Kør celle og bliv
- Variabler persisterer på tværs af celler

### Opsætning

```bash
# Installér Kotlin Jupyter kernel
pip install kotlin-jupyter-kernel
```

In [ ]:
println("Hello EV World!")

---

## Det Agentiske Landskab

<img src="images/Orloff-1600x1067.webp" alt="Professor Orloff" width="600" style="max-width: 100%;">
<br><br>

> An LLM is just a brain in a jar. 🧠

— [Simon Maple](https://www.linkedin.com/in/simonmaple/), Head of Developer Relations @ Tessl ([AI Native podcast](https://open.spotify.com/episode/1HYcUlgkCa9VwvVfuy3fyx?nd=1&dlsi=a77f69fabf8b46b1))

### Hvorfor kan LLM'er kode så godt nu?

**Kode har indbygget feedback.** Til forskel fra "god tekst" kan vi objektivt måle om kode virker:
- Kompilerer det? ✅/❌
- Består tests? ✅/❌
- Løser det opgaven? ✅/❌

Dette muliggør **RLVR** (*Reinforcement Learning with Verifiable Rewards*):

<img src="images/ai-training-loop.svg" alt="AI Træningsloop" width="600" style="max-width: 100%;">

### Benchmarks (pr. januar 2026)

#### SWE-bench: Løs rigtige GitHub issues

**SWE-bench Verified** (500 human-validerede issues):

| # | Model | Score |
|---|-------|-------|
| 1 | Claude Opus 4.5 | 80.9% |
| 2 | GPT-5.2 | 75.4% |
| 3 | Claude Opus 4.1 | 74.5% |

**SWE-bench Pro** (1865 sværere issues, nye repos):

| # | Agent | Score |
|---|-------|-------|
| 1 | Claude Opus 4.5 | 45.9% |
| 2 | Claude Sonnet 4.5 | 43.6% |
| 3 | Gemini 3 Pro Preview | 43.3% |

[swebench.com](https://www.swebench.com/)

---

#### METR: Hvor lang tid kan AI arbejde selvstændigt?

METR måler **tidshorisonten** - hvor lange opgaver kan AI løse med 50% succes?

| # | Model | 50% Tidshorisont |
|---|-------|------------------|
| 1 | Claude Opus 4.5 | ~4.8 timer |
| 2 | GPT-5.1 Codex Max | ~2.9 timer |
| 3 | GPT-5 | ~2.3 timer |

<br><img src="images/metr-timeline.svg" alt="METR Tidslinje" width="600" style="max-width: 100%;">

**Bemærk:**
- Tidshorisonten **fordobles hver 7. måned** (konsistent over 6 år, dvs. en pendant til Moore's Law)

**Hvis trenden fortsætter:**
- 2-4 år: ugelange opgaver
- Årtiet ud: månedlange projekter

[metr.org](https://metr.org/blog/2025-03-19-measuring-ai-ability-to-complete-long-tasks/)

---

#### DPAI Arena: Full developer workflow

JetBrains' benchmark - evaluerer hele engineering lifecycle på Spring-projekter (140+ opgaver).

| # | Agent | Model | Score |
|---|-------|-------|-------|
| 1 | Junie CLI 533.2 | Claude Opus 4.5 | 68.9 |
| 2 | Claude Code 2.0.55 | Claude Opus 4.5 | 68.0 |
| 3 | Junie CLI 496.3-prototype | Claude Sonnet 4.5 | 68.0 |

[dpaia.dev](https://dpaia.dev/)

---

### Men pas på: Benchmark ≠ virkelighed

METR-studie (juli 2025):
- Erfarne udviklere *troede* de var **20% hurtigere** med AI, når objektive tests viste de var **19% langsommere**

Sikkerhedssårbarheder ([Stanford/Boneh 2023](https://arxiv.org/abs/2211.03622)):
- Mennesker uden AI: **~50%** sårbar kode
- Mennesker MED AI: **~64%** — faktisk **værre**!

Forskning tager lang tid og der innoveres med lynets hast, så det handler i høj grad om at etablere et væddemål.

## Prerequisites Verification

### 1. Er Ollama installeret og kørende?

In [ ]:
import java.net.URL

val selectedModelName = "llama3.1:8b"

try {
    val response = URL("http://localhost:11434/api/tags").readText()
    println("✅ Ollama kører! Model: $selectedModelName")
} catch (e: Exception) {
    println("❌ Ollama kører IKKE. Start med: ollama serve")
}

---

# Del 1: Agenter - teori og byg din egen
*70 minutter inkl. øvelser*

> **"The LLM never actually touches your filesystem. It just *asks* for things to happen."**
> — Mihail Eric, ["The Emperor Has No Clothes"](https://www.mihaileric.com/The-Emperor-Has-No-Clothes/)

AI coding assistants virker som magi, men kernen er simpel:
- **~200 linjer kode** er alt der skal til
- LLM beslutter hvad der skal ske
- **Din kode** udfører handlingerne lokalt
- Resultater sendes tilbage til LLM

### The Agentic Loop - 4 Steps

<img src="images/agentic-loop.svg" alt="Agentic Loop" width="600" style="max-width: 100%;">

**Nøgleindsigt:** LLM'en kalder aldrig noget. Den outputter JSON der siger "Jeg vil gerne kalde X med Y". **Vi** eksekverer det!

## 1.1 Hvad LLM'en Faktisk Ser

Lad os se præcis hvad der sendes til Ollama.

In [ ]:
// Dette er PRÆCIS hvad vi sender til Ollama som tool definition. Det er bare JSON der beskriver en funktion.

val toolJsonExample = """
{
  "type": "function",
  "function": {
    "name": "get_station_status",
    "description": "Get real-time status of an EV charging station",
    "parameters": {
      "type": "object",
      "properties": {
        "stationId": {
          "type": "string",
          "description": "The station ID, e.g., CPH-001"
        }
      },
      "required": ["stationId"]
    }
  }
}
"""

println("Dette er hvad LLM'en ser som tool definition:")
println(toolJsonExample)

In [ ]:
// Og dette er hvad LLM'en returnerer når den vil bruge et tool

val toolCallResponseExample = """
{
  "message": {
    "role": "assistant",
    "content": "",
    "tool_calls": [
      {
        "function": {
          "name": "get_station_status",
          "arguments": {
            "stationId": "CPH-001"
          }
        }
      }
    ]
  }
}
"""

println("Når LLM'en vil kalde et tool, returnerer den:")
println(toolCallResponseExample)
println("\nLLM'en eksekverer ikke noget - den beder os om at gøre det!")

## 1.2 SimpleTool - Den Transparente Abstraktion

Nu bygger vi vores egen tool-definition.

**Forskellen på SimpleTool og rå JSON:**
- `SimpleTool` har en `execute` funktion - du kan se præcis hvad der kører
- Rå JSON er kun en data-struktur for serialisering, dvs. "automagi"

In [ ]:
// Vi bruger kotlinx.serialization JsonObject som parameter-type
// Men først definerer vi SimpleTool med en simpel (JsonObject) -> String execute funktion

// Minimal JSON parsing med standard library for nu
data class SimpleTool(
    val name: String,
    val description: String,
    val parameterSchema: String, // Raw JSON string
    val execute: (Map<String, Any?>) -> String
)

println("✅ SimpleTool defineret!")

## 1.3 ChargeSmart Domæne - Mock Data

Vi arbejder med et fiktivt EV-ladenetværk i København:

In [ ]:
import java.time.LocalDateTime

data class StationInfo(
    val id: String,
    val name: String,
    val location: String,
    val powerKw: Int,
    val type: String,
    val latitude: Double,
    val longitude: Double
)

data class SessionInfo(
    val vehicleId: String,
    val start: LocalDateTime,
    val kwh: Double
)

val stations = mapOf(
    "CPH-001" to StationInfo("CPH-001", "Nørreport Station", "Nørreport", 150, "ultra-fast", 55.6839, 12.5715),
    "CPH-002" to StationInfo("CPH-002", "Fisketorvet Parking", "Fisketorvet", 50, "fast", 55.6692, 12.5519),
    "CPH-003" to StationInfo("CPH-003", "Tivoli Garage", "Tivoli", 50, "fast", 55.6736, 12.5681),
    "CPH-004" to StationInfo("CPH-004", "Ørestad Center", "Ørestad", 150, "ultra-fast", 55.6310, 12.5770),
    "CPH-005" to StationInfo("CPH-005", "Amager Strandpark", "Amager", 22, "slow", 55.6520, 12.6050),
    "CPH-006" to StationInfo("CPH-006", "Nørrebro Runddel", "Nørrebro", 50, "fast", 55.7015, 12.5450),
    "CPH-007" to StationInfo("CPH-007", "Frederiksberg Have", "Frederiksberg", 7, "slow", 55.6784, 12.5293),
    "CPH-008" to StationInfo("CPH-008", "Kastrup Lufthavn P4", "Kastrup", 150, "ultra-fast", 55.6180, 12.6560),
    "CPH-009" to StationInfo("CPH-009", "Valby Langgade", "Valby", 22, "slow", 55.6631, 12.5120),
    "CPH-010" to StationInfo("CPH-010", "Hellerup Station", "Hellerup", 50, "fast", 55.7280, 12.5720)
)

val activeSessions = mapOf(
    "CPH-002" to SessionInfo("Tesla-Model-3", LocalDateTime.now().minusMinutes(45), 28.5),
    "CPH-004" to SessionInfo("Polestar-2", LocalDateTime.now().minusMinutes(12), 18.2),
    "CPH-007" to SessionInfo("VW-ID4", LocalDateTime.now().minusHours(2), 11.0)
)

fun getTariff(hour: Int): Double = when {
    hour in 0..5 -> 1.50
    hour in 6..16 -> 2.50
    hour in 17..20 -> 4.00
    else -> 2.50
}

println("✅ Loaded ${stations.size} stationer, ${activeSessions.size} aktive sessioner")

## 1.4 Øvelse 1: Forstå Execute-funktionerne til ChargeSmart Tools
*~15 minutter*

Nu skal du **forstå og evt. modificere** `execute`-funktionerne for 3 ChargeSmart tools.

**📋 Opgave:** Før du kører cellen nedenfor:
1. Læs koden igennem og forstå hvad hver funktion gør
2. Prøv at skjule koden og skrive din egen version (brug hints nedenfor)
3. Sammenlign med løsningen og kør cellen

**Du har adgang til:**
- `stations` - Map med alle stationer (ID → StationInfo)
- `activeSessions` - Map med aktive sessioner (stationId → SessionInfo)
- `getTariff(hour: Int)` - Returnerer pris pr. kWh for en given time

**Krav til de 3 tools:**

| Tool | Input | Output |
|------|-------|--------|
| `get_station_status` | stationId (string) | Status på én station (ledig/optaget, kW, type) |
| `list_stations` | *ingen* | Liste over alle stationer med status |
| `calculate_charging_cost` | kwh (number), hour (int) | Pris for at lade X kWh på tidspunkt Y |

In [ ]:
// 🎯 ØVELSE 1: Studer disse Execute-funktioner!
// Tip: args er et Map<String, Any?> - brug args["name"] as String osv.

val tools = mutableMapOf(
    "get_station_status" to SimpleTool(
        name = "get_station_status",
        description = "Get real-time status of an EV charging station including availability, power level, and current session info",
        parameterSchema = """{"type":"object","properties":{"stationId":{"type":"string","description":"Station ID like CPH-001"}},"required":["stationId"]}""",
        execute = { args ->
            // 1. Hent stationId fra args
            val stationId = args["stationId"] as? String ?: ""

            // 2. Slå op i stations map
            val station = stations[stationId]
            if (station == null) {
                "Error: Station $stationId not found"
            } else {
                // 3. Tjek om der er en aktiv session
                val session = activeSessions[stationId]
                if (session != null) {
                    "Station $stationId (${station.name}): IN USE by ${session.vehicleId} since ${session.start.toLocalTime().toString().take(5)}, ${"%.1f".format(session.kwh)} kWh delivered. ${station.powerKw}kW ${station.type} charger."
                } else {
                    "Station $stationId (${station.name}): AVAILABLE. ${station.powerKw}kW ${station.type} charger at ${station.location}."
                }
            }
        }
    ),

    "list_stations" to SimpleTool(
        name = "list_stations",
        description = "List all EV charging stations in the ChargeSmart network with their current availability",
        parameterSchema = """{"type":"object","properties":{}}""",
        execute = { _ ->
            val lines = stations.values.map { s ->
                val status = if (activeSessions.containsKey(s.id)) "IN USE" else "AVAILABLE"
                "- ${s.id}: ${s.name} (${s.location}) - ${s.powerKw}kW ${s.type} - $status"
            }
            "ChargeSmart Network Stations:\n" + lines.joinToString("\n")
        }
    ),

    "calculate_charging_cost" to SimpleTool(
        name = "calculate_charging_cost",
        description = "Calculate the cost of charging based on kWh needed and time of day",
        parameterSchema = """{"type":"object","properties":{"kwh":{"type":"number","description":"Energy needed in kWh"},"hour":{"type":"integer","description":"Hour of day (0-23)"}},"required":["kwh","hour"]}""",
        execute = { args ->
            // Handle both number and string JSON values from LLM
            val kwh = when (val v = args["kwh"]) {
                is Number -> v.toDouble()
                is String -> v.toDouble()
                else -> 0.0
            }
            val hour = when (val v = args["hour"]) {
                is Number -> v.toInt()
                is String -> v.toInt()
                else -> 12
            }
            val tariff = getTariff(hour)
            val cost = kwh * tariff

            val period = when {
                hour in 0..5 -> "off-peak"
                hour in 17..20 -> "peak"
                else -> "normal"
            }

            "Charging ${"%,.1f".format(kwh)} kWh at ${"%02d".format(hour)}:00 ($period tariff: ${"%,.2f".format(tariff)} DKK/kWh) = ${"%,.2f".format(cost)} DKK"
        }
    )
)

println("✅ Defineret ${tools.size} tools:")
tools.values.forEach { tool ->
    println("   - ${tool.name}: ${tool.description.take(50)}...")
}

### Hints til Øvelse 1

<details>
<summary>💡 Hint 1: Sådan læser du fra args Map</summary>

```kotlin
// Hent en string property
val stationId = args["stationId"] as? String ?: ""

// Hent et tal (kan være Number eller String fra JSON)
val kwh = when (val v = args["kwh"]) {
    is Number -> v.toDouble()
    is String -> v.toDouble()
    else -> 0.0
}
```

</details>

<details>
<summary>💡 Hint 2: get_station_status struktur</summary>

```kotlin
execute = { args ->
    val stationId = args["stationId"] as? String ?: ""

    val station = stations[stationId]
    if (station == null) {
        "Error: Station $stationId not found"
    } else {
        val session = activeSessions[stationId]
        // Returner forskellige beskeder baseret på session
        if (session != null)
            "Station $stationId: IN USE..."
        else
            "Station $stationId: AVAILABLE..."
    }
}
```

</details>

<details>
<summary>💡 Hint 3: list_stations med map/joinToString</summary>

```kotlin
execute = { _ ->
    val lines = stations.values.map { s ->
        val status = if (activeSessions.containsKey(s.id)) "IN USE" else "AVAILABLE"
        "- ${s.id}: ${s.name} - $status"
    }
    lines.joinToString("\n")
}
```

</details>

<details>
<summary>✅ Fuld løsning: Alle 3 tools</summary>

```kotlin
val tools = mutableMapOf(
    "get_station_status" to SimpleTool(
        name = "get_station_status",
        description = "Get real-time status of an EV charging station including availability, power level, and current session info",
        parameterSchema = """{"type":"object","properties":{"stationId":{"type":"string","description":"Station ID like CPH-001"}},"required":["stationId"]}""",
        execute = { args ->
            val stationId = args["stationId"] as? String ?: ""
            val station = stations[stationId]
            if (station == null) {
                "Error: Station $stationId not found"
            } else {
                val session = activeSessions[stationId]
                if (session != null) {
                    "Station $stationId (${station.name}): IN USE by ${session.vehicleId} since ${session.start.toLocalTime().toString().take(5)}, ${"%.1f".format(session.kwh)} kWh delivered. ${station.powerKw}kW ${station.type} charger."
                } else {
                    "Station $stationId (${station.name}): AVAILABLE. ${station.powerKw}kW ${station.type} charger at ${station.location}."
                }
            }
        }
    ),

    "list_stations" to SimpleTool(
        name = "list_stations",
        description = "List all EV charging stations in the ChargeSmart network with their current availability",
        parameterSchema = """{"type":"object","properties":{}}""",
        execute = { _ ->
            val lines = stations.values.map { s ->
                val status = if (activeSessions.containsKey(s.id)) "IN USE" else "AVAILABLE"
                "- ${s.id}: ${s.name} (${s.location}) - ${s.powerKw}kW ${s.type} - $status"
            }
            "ChargeSmart Network Stations:\n" + lines.joinToString("\n")
        }
    ),

    "calculate_charging_cost" to SimpleTool(
        name = "calculate_charging_cost",
        description = "Calculate the cost of charging based on kWh needed and time of day",
        parameterSchema = """{"type":"object","properties":{"kwh":{"type":"number","description":"Energy needed in kWh"},"hour":{"type":"integer","description":"Hour of day (0-23)"}},"required":["kwh","hour"]}""",
        execute = { args ->
            val kwh = when (val v = args["kwh"]) {
                is Number -> v.toDouble()
                is String -> v.toDouble()
                else -> 0.0
            }
            val hour = when (val v = args["hour"]) {
                is Number -> v.toInt()
                is String -> v.toInt()
                else -> 12
            }
            val tariff = getTariff(hour)
            val cost = kwh * tariff
            val period = when {
                hour in 0..5 -> "off-peak"
                hour in 17..20 -> "peak"
                else -> "normal"
            }
            "Charging ${"%,.1f".format(kwh)} kWh at ${"%%02d".format(hour)}:00 ($period tariff: ${"%,.2f".format(tariff)} DKK/kWh) = ${"%,.2f".format(cost)} DKK"
        }
    )
)
```

</details>

### Verificer din løsning

Kør cellen nedenfor for at teste dine Execute-funktioner **uden LLM**:

In [ ]:
// Test dine tools direkte!
println("=== Test get_station_status ===")
println(tools["get_station_status"]!!.execute(mapOf("stationId" to "CPH-001")))
println()

println("=== Test list_stations ===")
println(tools["list_stations"]!!.execute(emptyMap()))
println()

println("=== Test calculate_charging_cost ===")
println(tools["calculate_charging_cost"]!!.execute(mapOf("kwh" to 50.0, "hour" to 18)))
println()

println("✅ Alle tests kørte! Hvis du ser output ovenfor, virker dine tools.")

## 1.5 The Agentic Loop - det er det hele

Nu bygger vi det komplette agentic loop fra bunden. Vi bruger ktor HTTP client til at kalde Ollama's REST API direkte.

In [ ]:
@file:DependsOn("io.ktor:ktor-client-core:2.3.7")
@file:DependsOn("io.ktor:ktor-client-cio:2.3.7")
@file:DependsOn("io.ktor:ktor-client-content-negotiation:2.3.7")
@file:DependsOn("io.ktor:ktor-serialization-kotlinx-json:2.3.7")
@file:DependsOn("org.jetbrains.kotlinx:kotlinx-serialization-json:1.6.2")

import io.ktor.client.*
import io.ktor.client.engine.cio.*
import io.ktor.client.plugins.contentnegotiation.*
import io.ktor.client.request.*
import io.ktor.client.statement.*
import io.ktor.http.*
import io.ktor.serialization.kotlinx.json.*
import kotlinx.serialization.*
import kotlinx.serialization.json.*

println("✅ Dependencies loaded!")

In [ ]:
val client = HttpClient(CIO) {
    install(ContentNegotiation) {
        json(Json { ignoreUnknownKeys = true })
    }
}

val jsonParser = Json { ignoreUnknownKeys = true }

// Hjælpefunktion: Parse JsonObject arguments til Map<String, Any?>
fun jsonObjectToMap(obj: JsonObject): Map<String, Any?> {
    return obj.mapValues { (_, value) ->
        when (value) {
            is JsonPrimitive -> when {
                value.isString -> value.content
                value.content == "true" -> true
                value.content == "false" -> false
                value.content.contains(".") -> value.content.toDoubleOrNull()
                else -> value.content.toLongOrNull() ?: value.content
            }
            else -> value.toString()
        }
    }
}

println("✅ HTTP client og hjælpefunktioner klar!")

In [ ]:
// THE AGENTIC LOOP - ingen magi!

suspend fun runAgent(userMessage: String, tools: Map<String, SimpleTool>): String {
    val messages = mutableListOf(
        buildJsonObject {
            put("role", "system")
            put("content", """Du er en hjælpsom EV-ladnings assistent for ChargeSmart netværket i København.
                Brug de tilgængelige tools til at besvare spørgsmål om ladestationer,
                tilgængelighed, og priser. Svar altid på dansk.""")
        },
        buildJsonObject {
            put("role", "user")
            put("content", userMessage)
        }
    )

    val ollamaTools = tools.values.map { tool ->
        buildJsonObject {
            put("type", "function")
            putJsonObject("function") {
                put("name", tool.name)
                put("description", tool.description)
                put("parameters", jsonParser.parseToJsonElement(tool.parameterSchema))
            }
        }
    }

    val maxIterations = 10
    for (iteration in 1..maxIterations) {
        println("\n--- Iteration $iteration ---")

        val requestBody = buildJsonObject {
            put("model", selectedModelName)
            put("messages", JsonArray(messages))
            put("tools", JsonArray(ollamaTools))
            put("stream", false)
        }

        val response = client.post("http://localhost:11434/api/chat") {
            contentType(ContentType.Application.Json)
            setBody(requestBody.toString())
        }

        val responseJson = jsonParser.parseToJsonElement(response.bodyAsText()).jsonObject
        val assistantMessage = responseJson["message"]?.jsonObject ?: return "Error: No response"
        messages.add(assistantMessage)

        val toolCalls = assistantMessage["tool_calls"]?.jsonArray
        if (toolCalls == null || toolCalls.isEmpty()) {
            println("✅ LLM finished (no tool calls)")
            return assistantMessage["content"]?.jsonPrimitive?.content ?: "No response"
        }

        println("🔧 LLM wants to call ${toolCalls.size} tool(s)")

        for (toolCall in toolCalls) {
            val funcObj = toolCall.jsonObject["function"]?.jsonObject ?: continue
            val toolName = funcObj["name"]?.jsonPrimitive?.content ?: "unknown"
            val toolArgs = funcObj["arguments"]?.jsonObject ?: JsonObject(emptyMap())
            println("   Tool: $toolName")
            println("   Args: $toolArgs")

            val tool = tools[toolName]
            if (tool != null) {
                val argsMap = jsonObjectToMap(toolArgs)
                val result = tool.execute(argsMap)  // <-- DIN kode kører her!
                println("   📤 Result: ${result.take(80)}...")
                messages.add(buildJsonObject {
                    put("role", "tool")
                    put("content", result)
                })
            } else {
                messages.add(buildJsonObject {
                    put("role", "tool")
                    put("content", "Error: Unknown tool $toolName")
                })
            }
        }
        // Loop fortsætter - LLM ser resultaterne
    }

    return "Max iterations reached"
}

println("✅ runAgent function defined")

## 1.6 Øvelse 2: Human-in-the-Loop Confirmation
*~15 minutter*

I produktion vil du ofte have tools der gør "farlige" ting - f.eks. starter en ladesession, foretager en betaling, eller sletter data. Før sådanne tools eksekveres, bør agenten **bede om bekræftelse**.

**Scenarie:** Forestil dig at `calculate_charging_cost` kunne udløse en forhåndsreservation eller pre-authorization på dit kort. Før vi eksekverer det, vil vi gerne have brugerens godkendelse.

**Din opgave:** Modificer loopet så det beder om bekræftelse før "farlige" tools eksekveres.

In [ ]:
// 🎯 ØVELSE 2: Tilføj human-in-the-loop confirmation

// Definer hvilke tools der kræver bekræftelse
val dangerousTools = setOf("calculate_charging_cost")

// RunAgentWithConfirmation - næsten identisk med runAgent!
// Bruger readLine() til at få bruger-input
suspend fun runAgentWithConfirmation(
    userMessage: String,
    tools: Map<String, SimpleTool>
): String {
    val messages = mutableListOf(
        buildJsonObject {
            put("role", "system")
            put("content", """Du er en hjælpsom EV-ladnings assistent for ChargeSmart netværket i København.
                Brug de tilgængelige tools til at besvare spørgsmål om ladestationer,
                tilgængelighed, og priser. Svar altid på dansk.""")
        },
        buildJsonObject {
            put("role", "user")
            put("content", userMessage)
        }
    )

    val ollamaTools = tools.values.map { tool ->
        buildJsonObject {
            put("type", "function")
            putJsonObject("function") {
                put("name", tool.name)
                put("description", tool.description)
                put("parameters", jsonParser.parseToJsonElement(tool.parameterSchema))
            }
        }
    }

    val maxIterations = 10
    for (iteration in 1..maxIterations) {
        println("\n--- Iteration $iteration ---")

        val requestBody = buildJsonObject {
            put("model", selectedModelName)
            put("messages", JsonArray(messages))
            put("tools", JsonArray(ollamaTools))
            put("stream", false)
        }

        val response = client.post("http://localhost:11434/api/chat") {
            contentType(ContentType.Application.Json)
            setBody(requestBody.toString())
        }

        val responseJson = jsonParser.parseToJsonElement(response.bodyAsText()).jsonObject
        val assistantMessage = responseJson["message"]?.jsonObject ?: return "Error: No response"
        messages.add(assistantMessage)

        val toolCalls = assistantMessage["tool_calls"]?.jsonArray
        if (toolCalls == null || toolCalls.isEmpty()) {
            println("✅ LLM finished (no tool calls)")
            return assistantMessage["content"]?.jsonPrimitive?.content ?: "No response"
        }

        println("🔧 LLM wants to call ${toolCalls.size} tool(s)")

        for (toolCall in toolCalls) {
            val funcObj = toolCall.jsonObject["function"]?.jsonObject ?: continue
            val toolName = funcObj["name"]?.jsonPrimitive?.content ?: "unknown"
            val toolArgs = funcObj["arguments"]?.jsonObject ?: JsonObject(emptyMap())
            println("   Tool: $toolName")
            println("   Args: $toolArgs")

            // 🎯 ØVELSE: Bekræftelseslogik!
            if (toolName in dangerousTools) {
                print("⚠️ Execute '$toolName'? (y/n): ")
                val confirmation = readLine()
                if (confirmation?.trim()?.lowercase() != "y") {
                    println("   ⏭️ Skipped by user")
                    messages.add(buildJsonObject {
                        put("role", "tool")
                        put("content", "Tool $toolName was skipped by user")
                    })
                    continue
                }
                println("   ✅ Confirmed by user")
            }

            val tool = tools[toolName]
            if (tool != null) {
                val argsMap = jsonObjectToMap(toolArgs)
                val result = tool.execute(argsMap)
                println("   📤 Result: ${result.take(80)}...")
                messages.add(buildJsonObject {
                    put("role", "tool")
                    put("content", result)
                })
            } else {
                messages.add(buildJsonObject {
                    put("role", "tool")
                    put("content", "Error: Unknown tool $toolName")
                })
            }
        }
    }

    return "Max iterations reached"
}

println("✅ runAgentWithConfirmation function defined")

### Om readLine()

`readLine()` bruges til at få bruger-input i Kotlin notebooks. I Jupyter vises en input dialog.

<details>
<summary>💡 Reference: Bekræftelseslogikken</summary>

```kotlin
// Tjek om tool er "farligt" og bed om bekræftelse
if (toolName in dangerousTools) {
    print("⚠️ Execute '$toolName'? (y/n): ")
    val confirmation = readLine()
    if (confirmation?.trim()?.lowercase() != "y") {
        println("   ⏭️ Skipped by user")
        messages.add(buildJsonObject {
            put("role", "tool")
            put("content", "Tool $toolName was skipped by user")
        })
        continue  // Skip til næste tool call
    }
    println("   ✅ Confirmed by user")
}
```

**Lærdom:** Human-in-the-loop er kritisk for produktions-agents. Du bestemmer hvilke handlinger der kræver godkendelse!

</details>

### Test din løsning

Kør cellen nedenfor. Når LLM vil kalde `calculate_charging_cost`, vil Jupyter vise en input dialog. Skriv `y` for at godkende eller `n` for at afvise.

In [ ]:
// Test human-in-the-loop - Jupyter vil vise en input dialog!
import kotlinx.coroutines.runBlocking

runBlocking {
    val result = runAgentWithConfirmation(
        "Hvad koster det at lade 50 kWh kl. 18?",
        tools
    )

    println("\n=== Final Response ===")
    println(result)
}

## 1.7 Test din Agent!

In [ ]:
import kotlinx.coroutines.runBlocking

runBlocking {
    val result = runAgent(
        "Hvilke ladestationer er ledige lige nu?",
        tools
    )

    println("\n=== Final Response ===")
    println(result)
}

In [ ]:
import kotlinx.coroutines.runBlocking

runBlocking {
    val result2 = runAgent(
        "Hvad er status på CPH-002?",
        tools
    )

    println("\n=== Final Response ===")
    println(result2)
}

In [ ]:
import kotlinx.coroutines.runBlocking

runBlocking {
    val result3 = runAgent(
        "Hvad koster det at lade 50 kWh kl. 18? Og hvad hvis jeg venter til kl. 22?",
        tools
    )

    println("\n=== Final Response ===")
    println(result3)
}

## Checkpoint Del 1

Du har nu:
1. ✅ Set præcis hvad LLM'en modtager (JSON tool definitions)
2. ✅ Set præcis hvad LLM'en returnerer (JSON tool calls)
3. ✅ Bygget `SimpleTool` - en transparent tool abstraction
4. ✅ **Øvelse 1:** Skrevet Execute-funktioner til 3 ChargeSmart tools
5. ✅ Set det komplette agentic loop med ktor HTTP client
6. ✅ **Øvelse 2:** Tilføjet human-in-the-loop confirmation til loopet
7. ✅ Kørt en agent med dine tools

**Nøgleindsigt:** Det er så simpelt. Et loop + lidt kode per tool. *The emperor has no clothes!*

**Lærdom fra Øvelse 2:** I produktion er human-in-the-loop kritisk. Du bestemmer hvilke handlinger der kræver godkendelse!

---

# Del 2: Abstraktionsstigen – SimpleTool vs rå JSON
*15 minutter (valgfri)*

Nu hvor du forstår hvad der sker "under the hood", lad os se forskellen på vores SimpleTool og rå JSON.

## 2.1 Side-by-Side: SimpleTool vs rå JSON

In [ ]:
// DIN SimpleTool - alt er synligt:
val simpleToolExample = SimpleTool(
    name = "get_station_status",
    description = "Get EV charging station status",
    parameterSchema = """{"type":"object","properties":{"stationId":{"type":"string"}},"required":["stationId"]}""",
    execute = { args -> "Station ${args["stationId"]} is available" }  // <-- Koden er HER
)

// Rå JSON tool format - samme data, men ingen execute:
val rawJsonToolExample = buildJsonObject {
    put("type", "function")
    putJsonObject("function") {
        put("name", "get_station_status")
        put("description", "Get EV charging station status")
        putJsonObject("parameters") {
            put("type", "object")
            putJsonObject("properties") {
                putJsonObject("stationId") {
                    put("type", "string")
                    put("description", "Station ID")
                }
            }
            put("required", JsonArray(listOf(JsonPrimitive("stationId"))))
        }
    }
}
// BEMÆRK: Ingen Execute! Du skal håndtere eksekveringen separat i dit loop.

println("SimpleTool har:")
println("  - name, description, parameterSchema")
println("  - execute: (Map<String, Any?>) -> String ← KODEN ER HER")
println()
println("Rå JSON tool har:")
println("  - type, function (name, description, parameters)")
println("  - Ingen execute! ← Du skal selv matche tool calls til kode")

## 2.2 Nøgleindsigt

**Rå JSON tool er bare en data-struktur til JSON-serialisering.**

Det er derfor vi byggede `SimpleTool` først - for at se det fulde billede:
1. Tool *definition* (JSON schema) → sendes til LLM
2. Tool *execution* (din kode) → kører lokalt når LLM requester det

Libraries giver dig #1, men du skal selv håndtere #2.
Vores `SimpleTool` pakker begge dele sammen så det er tydeligt.

## 2.3 Øvelse: Refaktorer til rå JSON format

Konverter én af dine SimpleTool definitioner til rå JSON tool format.
Behold samme execution logic i dit loop.

<details>
<summary>Hint: Se løsningen</summary>

```kotlin
// SimpleTool version:
val getStatus = SimpleTool(
    name = "get_station_status",
    // ...
    execute = { args -> /* kode */ }
)

// Rå JSON version:
val getStatusTool = buildJsonObject {
    put("type", "function")
    putJsonObject("function") {
        put("name", "get_station_status")
        // ... (ingen execute!)
    }
}

// Execution håndteres separat i loop:
if (toolName == "get_station_status") {
    result = /* din execution kode */
}
```

</details>

---

# ☕ PAUSE (15 min)

---

# Del 3: Abstraktionsstigen – MCP
*45 minutter inkl. øvelse*

Vi fortsætter op ad abstraktionsstigen: SimpleTool → Rå JSON → **MCP**.

## Teori: Model Context Protocol (MCP)

### Problemet: N×M Integrationer

<img src="images/mcp-problem.svg" alt="MCP Problem" width="600" style="max-width: 100%;">

Uden standardisering: Hver agent skal have custom integration til hver tool.

### Løsningen: MCP som Standard

<img src="images/mcp-solution.svg" alt="MCP Solution" width="600" style="max-width: 100%;">

**MCP = USB-C for AI tools.** *Plug any tool into any agent.*

## 3.1 Hvad Ændrer Sig med MCP?

**Før (Del 1-2):** Tools defineret inline i din kode
```kotlin
val tools = mutableMapOf(
    "get_station_status" to SimpleTool(..., execute = { args -> ... })
)
// Tools lever i din agent kode
```

**Efter (Del 3):** Tools kommer fra eksterne MCP servere
```kotlin
val mcpClient = McpClient.connect(transport)
val tools = mcpClient.listTools()
// Tools opdages dynamisk fra server!
```

**Samme agent loop, men tools er *"pluggable"*.**

## 3.2 MCP Protocol Basics

MCP bruger **JSON-RPC 2.0** over stdio (lokalt) eller SSE (remote).

Tre hovedoperationer:
1. `initialize` - Handshake mellem client og server
2. `tools/list` - Hent liste af tilgængelige tools
3. `tools/call` - Kald et specifikt tool

## 3.3 Øvelse 3: Higher-level Tools med Kotlin funktioner
*~10 minutter*

MCP er designet til **inter-proces** kommunikation (client og server i separate processer).
I en notebook bruger vi i stedet typed Kotlin funktioner, som giver samme tool-oplevelse **in-process**.

> **Note:** I produktion ville du forbinde til en ekstern MCP server via stdio eller HTTP transport.

**📋 Opgave:** Sammenlign `TypedTool` med `SimpleTool`:
- I Øvelse 1 brugte du `SimpleTool` med `execute: (Map<String, Any?>) -> String`
- Her bruger vi `TypedTool` med mere strukturerede parameter-definitioner
- Parametrene er stadig JSON-baserede, men toolet er mere selvbeskrivende

**Bemærk:** Logikken er den samme - kun signaturen er anderledes.

In [ ]:
// 🎯 ØVELSE 3: Opret higher-level tools med typed Kotlin funktioner

// Vi wrapper vores tools i en mere struktureret form
data class TypedTool(
    val name: String,
    val description: String,
    val parameterSchema: String,
    val execute: (Map<String, Any?>) -> String
)

val aiTools = mutableListOf(
    // get_station_status - givet som eksempel
    TypedTool(
        name = "get_station_status",
        description = "Get real-time status of an EV charging station. Parameter: stationId (e.g. CPH-001)",
        parameterSchema = """{"type":"object","properties":{"stationId":{"type":"string"}},"required":["stationId"]}""",
        execute = { args ->
            val stationId = args["stationId"] as? String ?: ""
            val station = stations[stationId]
            if (station == null) {
                "Error: Station $stationId not found"
            } else {
                val session = activeSessions[stationId]
                if (session != null) {
                    "Station $stationId (${station.name}): IN USE by ${session.vehicleId} since ${session.start.toLocalTime().toString().take(5)}, ${"%.1f".format(session.kwh)} kWh delivered. ${station.powerKw}kW ${station.type} charger."
                } else {
                    "Station $stationId (${station.name}): AVAILABLE. ${station.powerKw}kW ${station.type} charger at ${station.location}."
                }
            }
        }
    ),

    // list_stations - studer denne implementering
    TypedTool(
        name = "list_stations",
        description = "List all EV charging stations in the ChargeSmart network with their current availability",
        parameterSchema = """{"type":"object","properties":{}}""",
        execute = { _ ->
            val lines = stations.values.map { s ->
                val status = if (activeSessions.containsKey(s.id)) "IN USE" else "AVAILABLE"
                "- ${s.id}: ${s.name} (${s.location}) - ${s.powerKw}kW ${s.type} - $status"
            }
            "ChargeSmart Network Stations:\n" + lines.joinToString("\n")
        }
    ),

    // calculate_charging_cost - studer denne implementering
    TypedTool(
        name = "calculate_charging_cost",
        description = "Calculate the cost of charging based on kWh needed and hour of day (0-23)",
        parameterSchema = """{"type":"object","properties":{"kwh":{"type":"number","description":"Energy needed in kWh"},"hour":{"type":"integer","description":"Hour of day (0-23)"}},"required":["kwh","hour"]}""",
        execute = { args ->
            val kwh = when (val v = args["kwh"]) {
                is Number -> v.toDouble()
                is String -> v.toDouble()
                else -> 0.0
            }
            val hour = when (val v = args["hour"]) {
                is Number -> v.toInt()
                is String -> v.toInt()
                else -> 12
            }
            val tariff = getTariff(hour)
            val cost = kwh * tariff
            val period = when {
                hour in 0..5 -> "off-peak"
                hour in 17..20 -> "peak"
                else -> "normal"
            }
            "Charging ${"%,.1f".format(kwh)} kWh at ${"%02d".format(hour)}:00 ($period tariff: ${"%,.2f".format(tariff)} DKK/kWh) = ${"%,.2f".format(cost)} DKK"
        }
    )
)

println("🔌 Oprettet ${aiTools.size} TypedTool tools:")
aiTools.forEach { println("   - ${it.name}") }

### Hints til Øvelse 3

<details>
<summary>💡 Hint: list_stations</summary>

```kotlin
TypedTool(
    name = "list_stations",
    description = "List all EV charging stations...",
    parameterSchema = """{"type":"object","properties":{}}""",
    execute = { _ ->
        val lines = stations.values.map { s ->
            val status = if (activeSessions.containsKey(s.id)) "IN USE" else "AVAILABLE"
            "- ${s.id}: ${s.name} (${s.location}) - ${s.powerKw}kW ${s.type} - $status"
        }
        "ChargeSmart Network Stations:\n" + lines.joinToString("\n")
    }
)
```

</details>

<details>
<summary>💡 Hint: calculate_charging_cost</summary>

```kotlin
TypedTool(
    name = "calculate_charging_cost",
    description = "Calculate the cost of charging...",
    parameterSchema = """{"type":"object","properties":{"kwh":{"type":"number"},"hour":{"type":"integer"}},"required":["kwh","hour"]}""",
    execute = { args ->
        val kwh = when (val v = args["kwh"]) {
            is Number -> v.toDouble()
            is String -> v.toDouble()
            else -> 0.0
        }
        val hour = when (val v = args["hour"]) {
            is Number -> v.toInt()
            is String -> v.toInt()
            else -> 12
        }
        val tariff = getTariff(hour)
        val cost = kwh * tariff
        val period = when {
            hour in 0..5 -> "off-peak"
            hour in 17..20 -> "peak"
            else -> "normal"
        }
        "Charging ${"%,.1f".format(kwh)} kWh at ${"%%02d".format(hour)}:00 ($period tariff: ${"%,.2f".format(tariff)} DKK/kWh) = ${"%,.2f".format(cost)} DKK"
    }
)
```

</details>

<details>
<summary>✅ Fuld løsning: Alle 3 TypedTool tools</summary>

```kotlin
val aiTools = mutableListOf(
    TypedTool(
        name = "get_station_status",
        description = "Get real-time status of an EV charging station. Parameter: stationId (e.g. CPH-001)",
        parameterSchema = """{"type":"object","properties":{"stationId":{"type":"string"}},"required":["stationId"]}""",
        execute = { args ->
            val stationId = args["stationId"] as? String ?: ""
            val station = stations[stationId]
            if (station == null) "Error: Station $stationId not found"
            else {
                val session = activeSessions[stationId]
                if (session != null)
                    "Station $stationId (${station.name}): IN USE by ${session.vehicleId}..."
                else
                    "Station $stationId (${station.name}): AVAILABLE..."
            }
        }
    ),
    TypedTool(
        name = "list_stations",
        description = "List all EV charging stations in the ChargeSmart network with their current availability",
        parameterSchema = """{"type":"object","properties":{}}""",
        execute = { _ ->
            val lines = stations.values.map { s ->
                val status = if (activeSessions.containsKey(s.id)) "IN USE" else "AVAILABLE"
                "- ${s.id}: ${s.name} (${s.location}) - ${s.powerKw}kW ${s.type} - $status"
            }
            "ChargeSmart Network Stations:\n" + lines.joinToString("\n")
        }
    ),
    TypedTool(
        name = "calculate_charging_cost",
        description = "Calculate the cost of charging based on kWh needed and hour of day (0-23)",
        parameterSchema = """{"type":"object","properties":{"kwh":{"type":"number"},"hour":{"type":"integer"}},"required":["kwh","hour"]}""",
        execute = { args ->
            val kwh = (args["kwh"] as? Number)?.toDouble() ?: 0.0
            val hour = (args["hour"] as? Number)?.toInt() ?: 12
            val tariff = getTariff(hour)
            val cost = kwh * tariff
            val period = when { hour in 0..5 -> "off-peak"; hour in 17..20 -> "peak"; else -> "normal" }
            "Charging $kwh kWh at $hour:00 ($period: $tariff DKK/kWh) = $cost DKK"
        }
    )
)
```

</details>

## 3.4 Automatiseret Tool Calling

Her kommer automationen af det loop du byggede i Del 1!

Vi laver en hjælpefunktion der bruger `TypedTool` listen og automatisk kalder tools.

In [ ]:
// Automatiseret agent med TypedTool
suspend fun runAgentWithTypedTools(userMessage: String, aiTools: List<TypedTool>): String {
    // Konverter TypedTool til tool map for genbrug
    val toolMap = aiTools.associateBy { it.name }
    val simpleToolMap = toolMap.mapValues { (_, typed) ->
        SimpleTool(
            name = typed.name,
            description = typed.description,
            parameterSchema = typed.parameterSchema,
            execute = typed.execute
        )
    }.toMutableMap()

    return runAgent(userMessage, simpleToolMap)
}

println("✅ runAgentWithTypedTools klar!")

## 3.5 Test: ChargeSmart med TypedTool

Kør samme query som i Del 1, men nu med TypedTool og automatiseret loop!

**Bemærk:** Samme tools, samme data - men via TypedTool abstraktionen.

In [ ]:
import kotlinx.coroutines.runBlocking

println("Sender besked med TypedTool tools...\n")

runBlocking {
    val result = runAgentWithTypedTools(
        "Hvilke ladestationer er ledige lige nu? Og hvad koster det at lade 30 kWh kl. 18?",
        aiTools
    )

    println("\n=== Final Response ===")
    println(result)
}

println("\n✅ Done!")

## Checkpoint Del 3

Du har nu:
1. ✅ Forstået MCP som "USB-C for AI tools"
2. ✅ **Øvelse 3:** Oprettet tools med `TypedTool` - en higher-level abstraction
3. ✅ Set hvordan automatisering wrapper loop'et fra Del 1
4. ✅ Kørt en agent med automatiseret tool invocation

**Nøgleindsigt:** MCP standardiserer tool connectivity. I produktion forbinder du til eksterne MCP servere - i notebooks bruger vi TypedTool som in-process alternativ.

---

# Del 3.5: Øvelse 4 - Udvid med get_nearest_station
*~20 minutter*

Nu skal du **studere hvordan man udvider** agenten med et helt nyt tool! Dette er den vigtigste øvelse, fordi det simulerer det typiske workflow: du har en fungerende agent og skal tilføje ny funktionalitet.

## Scenarie

En bruger står på gaden og vil finde den **nærmeste ledige ladestation**. De kender deres GPS-koordinater og vil have en anbefaling.

**📋 Opgave:** Studer implementeringen nedenfor og forstå logikken:
1. Modtag brugerens latitude og longitude
2. Filtrer alle **ledige** stationer (ikke optaget)
3. Beregn afstanden til hver station med Haversine
4. Returner den nærmeste

## Stationskoordinater

Alle stationer har nu latitude/longitude (vi tilføjede dem i starten):

| Station | Lokation | Latitude | Longitude |
|---------|----------|----------|-----------|
| CPH-001 | Nørreport | 55.6839 | 12.5715 |
| CPH-002 | Fisketorvet | 55.6692 | 12.5519 |
| CPH-003 | Tivoli | 55.6736 | 12.5681 |
| CPH-004 | Ørestad | 55.6310 | 12.5770 |
| CPH-005 | Amager | 55.6520 | 12.6050 |
| CPH-006 | Nørrebro | 55.7015 | 12.5450 |
| CPH-007 | Frederiksberg | 55.6784 | 12.5293 |
| CPH-008 | Kastrup | 55.6180 | 12.6560 |
| CPH-009 | Valby | 55.6631 | 12.5120 |
| CPH-010 | Hellerup | 55.7280 | 12.5720 |

In [ ]:
import kotlin.math.*

// Haversine-formlen til at beregne afstand mellem to koordinater (i km)
fun calculateDistance(lat1: Double, lon1: Double, lat2: Double, lon2: Double): Double {
    val R = 6371.0 // Jordens radius i km
    val dLat = Math.toRadians(lat2 - lat1)
    val dLon = Math.toRadians(lon2 - lon1)
    val a = sin(dLat / 2).pow(2) +
            cos(Math.toRadians(lat1)) * cos(Math.toRadians(lat2)) *
            sin(dLon / 2).pow(2)
    val c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c
}

// Test: Afstand fra Rådhuspladsen (55.6761, 12.5683) til Nørreport (CPH-001)
val testDistance = calculateDistance(55.6761, 12.5683, 55.6839, 12.5715)
println("Test: Afstand fra Rådhuspladsen til Nørreport: ${"%,.2f".format(testDistance)} km")

In [ ]:
// 🎯 DIN TUR: Implementer get_nearest_station

val getNearestStation = TypedTool(
    name = "get_nearest_station",
    description = "Find the nearest available charging station. Parameters: latitude and longitude of user's current position",
    parameterSchema = """{"type":"object","properties":{"latitude":{"type":"number","description":"User latitude"},"longitude":{"type":"number","description":"User longitude"}},"required":["latitude","longitude"]}""",
    execute = { args ->
        val latitude = when (val v = args["latitude"]) {
            is Number -> v.toDouble()
            is String -> v.toDouble()
            else -> 0.0
        }
        val longitude = when (val v = args["longitude"]) {
            is Number -> v.toDouble()
            is String -> v.toDouble()
            else -> 0.0
        }

        // TODO: 1. Filtrer ledige stationer fra stations.values
        //       2. Find nærmeste med calculateDistance()
        //       3. Returner beskrivende streng

        throw NotImplementedError("Implementer get_nearest_station!")
    }
)

// Tilføj til aiTools
aiTools.add(getNearestStation)
println("🔌 aiTools har nu ${aiTools.size} tools (inkl. get_nearest_station stub)")

### Hints til Øvelse 4

<details>
<summary>💡 Hint 1: Filtrer ledige stationer</summary>

```kotlin
val availableStations = stations.values
    .filter { !activeSessions.containsKey(it.id) }

if (availableStations.isEmpty())
    return@TypedTool "Ingen ledige stationer lige nu!"
```

</details>

<details>
<summary>💡 Hint 2: Beregn afstande og find minimum</summary>

```kotlin
val nearest = availableStations
    .map { s -> s to calculateDistance(latitude, longitude, s.latitude, s.longitude) }
    .minByOrNull { it.second }!!
```

</details>

<details>
<summary>✅ Fuld løsning</summary>

```kotlin
val getNearestStation = TypedTool(
    name = "get_nearest_station",
    description = "Find the nearest available charging station. Parameters: latitude and longitude of user's current position",
    parameterSchema = """{"type":"object","properties":{"latitude":{"type":"number"},"longitude":{"type":"number"}},"required":["latitude","longitude"]}""",
    execute = { args ->
        val latitude = when (val v = args["latitude"]) {
            is Number -> v.toDouble()
            is String -> v.toDouble()
            else -> 0.0
        }
        val longitude = when (val v = args["longitude"]) {
            is Number -> v.toDouble()
            is String -> v.toDouble()
            else -> 0.0
        }

        // Filtrer ledige stationer
        val availableStations = stations.values
            .filter { !activeSessions.containsKey(it.id) }

        if (availableStations.isEmpty()) {
            "Ingen ledige stationer lige nu!"
        } else {
            // Find nærmeste
            val nearest = availableStations
                .map { s -> s to calculateDistance(latitude, longitude, s.latitude, s.longitude) }
                .minByOrNull { it.second }!!

            "Nærmeste ledige station: ${nearest.first.id} (${nearest.first.name}) - " +
                "${"%,.2f".format(nearest.second)} km væk. " +
                "${nearest.first.powerKw}kW ${nearest.first.type} charger ved ${nearest.first.location}."
        }
    }
)
```

</details>

### Test dit nye tool

In [ ]:
import kotlinx.coroutines.runBlocking

// Test get_nearest_station med en position nær Rådhuspladsen
println("=== Test fra Rådhuspladsen (55.6761, 12.5683) ===\n")

runBlocking {
    val result = runAgentWithTypedTools(
        "Jeg står på Rådhuspladsen (koordinater: 55.6761, 12.5683). Hvor er den nærmeste ledige ladestation?",
        aiTools
    )

    println("\n=== Final Response ===")
    println(result)
}

println("\n✅ Test færdig!")

## Checkpoint Øvelse 4

Du har nu:
1. ✅ Forstået Haversine-formlen til afstandsberegning
2. ✅ Implementeret `get_nearest_station` med filter og afstandssortering
3. ✅ Udvidet en eksisterende agent med ny funktionalitet
4. ✅ Testet med en realistisk brugerforespørgsel

**Lærdom:** At udvide en agent er ofte nemmere end at bygge en fra bunden. Du tilføjer bare et nyt tool til listen!

---

# Del 4: Skills – teori og prøv med GitHub Copilot
*25 minutter*

## Fra domæne-specifikke agents til universel kode

**Hvordan vi plejede at tænke:**

<img src="images/anthropic-how-we-used-to-think-about-agents.png" alt="Separate agents for each domain" width="600" style="max-width: 100%;">

Vi troede hver domæne kræver sin egen agent med egne tools og scaffolding.

**Hvad vi opdagede:**

<img src="images/anthropic-code-is-the-universal-interface.png" alt="Code is the universal interface" width="600" style="max-width: 100%;">

Kode er ikke bare en use case – det er den universelle grænseflade til den digitale verden.

En coding agent kan:
- Kalde API'er for at hente data (research)
- Organisere data i filsystemet (dokumenter)
- Analysere med Python (finans)
- Generere output i ethvert format (marketing)

**Konklusion**: Vi behøver ikke mange forskellige agents – vi behøver ÉN general-purpose coding agent + **skills** der giver domæneekspertise.

## Teori: Hvorfor Skills?

**Problemet:** Giv en model 100 tools, og den bliver forvirret. Tool selection degraderer.

**Løsningen:** Skills = "lazy-loaded expertise"
- En "router" klassificerer først brugerens intent
- Kun relevante tools/knowledge indlæses
- Modellen fokuserer på én opgave ad gangen

**Key insight:** Copilot har allerede loopet, LLM, og basale tools. Du tilføjer domæneviden!

## 4.1 GitHub Copilot Agent Mode

VS Code 1.108+ har eksperimentel support for "agent skills".

### Aktivering

1. Åbn VS Code Settings (Ctrl+,)
2. Søg efter `chat.useAgentSkills`
3. Aktiver indstillingen

### Skill Struktur

```
.github/skills/
└── ev-charging-advisor/
    └── SKILL.md
```

Copilot scanner automatisk `.github/skills/` og indlæser relevante skills.

## 4.2 SKILL.md Anatomi

```yaml
---
name: my-skill
description: Kort beskrivelse til discovery
triggers:
  - "når bruger spørger om X"
  - "når der er behov for Y"
---

# Detaljerede Instruktioner

Her kommer den fulde domæneviden...
```

- **YAML frontmatter:** Metadata til skill discovery
- **Markdown body:** Instruktioner, eksempler, context

## 4.3 Øvelse: Opret EV Charging Advisor Skill

### Step 1: Opret skill mappe

**PowerShell:**
```powershell
New-Item -ItemType Directory -Force -Path ".github/skills/ev-charging-advisor"
```

**Bash:**
```bash
mkdir -p .github/skills/ev-charging-advisor
```

### Step 2: Opret SKILL.md

Opret filen `.github/skills/ev-charging-advisor/SKILL.md`:

```yaml
---
name: ev-charging-advisor
description: Ekspertrådgivning om optimering af elbil-ladning på ChargeSmart netværket
triggers:
  - "når bruger spørger om ladepris"
  - "når bruger vil finde billigste ladetidspunkt"
  - "når bruger spørger om batteri-tips"
  - "når bruger nævner ChargeSmart eller elbil-ladning"
---

# EV Charging Advisor

Du er en ekspert i at optimere elbil-ladning på ChargeSmart netværket i København.

## ChargeSmart Tarif-struktur

| Periode | Tid | Pris/kWh |
|---------|-----|----------|
| Off-peak | 00:00-06:00 | 1.50 DKK |
| Normal | 06:00-17:00, 21:00-00:00 | 2.50 DKK |
| Peak | 17:00-21:00 | 4.00 DKK |

## Stationer i Netværket

- CPH-001: Nørreport Station - 150kW ultra-fast
- CPH-002: Fisketorvet Parking - 50kW fast
- CPH-003: Tivoli Garage - 50kW fast
- CPH-004: Ørestad Center - 150kW ultra-fast
- CPH-005: Amager Strandpark - 22kW slow
- CPH-006: Nørrebro Runddel - 50kW fast
- CPH-007: Frederiksberg Have - 7kW slow
- CPH-008: Kastrup Lufthavn P4 - 150kW ultra-fast
- CPH-009: Valby Langgade - 22kW slow
- CPH-010: Hellerup Station - 50kW fast

## Instruktioner

Når bruger spørger om ladning:

1. **Identificer behov** - Hvor mange kWh skal de lade?
2. **Tjek tidspunkt** - Hvornår vil de lade?
3. **Beregn pris** - Brug tarif-tabellen
4. **Optimer** - Foreslå ALTID billigere alternativer
5. **Batteri-tips** - Tilføj relevante tips

## Batteri-tips

- Undgå at lade til 100% dagligt (80% er optimalt for battery health)
- Undgå at køre under 20% regelmæssigt
- Forvarm batteri i koldt vejr før hurtigladning
- Ultra-fast ladning (150kW) slider mere på batteriet end langsom ladning

## Eksempel-dialog

**Bruger:** Hvad koster det at lade 40 kWh kl. 18?

**Svar:** At lade 40 kWh kl. 18:00 koster **160 DKK** (peak-tarif: 4.00 DKK/kWh).

💡 **Tip:** Hvis du kan vente til kl. 21:00, falder prisen til **100 DKK** (normal-tarif).
Eller endnu bedre - lad natten over (kl. 00-06) for kun **60 DKK**!

🔋 **Batteri-tip:** Hvis du ikke behøver fuld opladning, så lad kun til 80% for at forlænge batteriets levetid.
```

### Step 3: Test med Copilot Chat

1. Åbn **Copilot Chat** i VS Code (Ctrl+Alt+I)
2. Prøv disse prompts:
   - "Hvad koster det at lade 50 kWh kl. 18?"
   - "Find den billigste tid at lade min elbil"
   - "Giv mig tips til at forlænge mit EV batteri"

Copilot burde nu bruge din skill til at give ChargeSmart-specifikke svar! (Det er tilfælded hvis du ser beskeden "Read SKILL.md file".)

## Checkpoint Del 4

Du har nu:
1. ✅ Forstået Skills som "lazy-loaded expertise"
2. ✅ Oprettet en SKILL.md med domæneviden
3. ✅ Udvidet GitHub Copilot med ChargeSmart ekspertise

**Nøgleindsigt:** Du behøver ikke bygge en agent - du kan udvide én du allerede har!

---

# Afrunding
*10 minutter*

## Key Takeaways

1. **Agents er simple** - ~200 linjer loop kode
2. **"Magien" er i LLM'en** - ikke infrastrukturen
3. **MCP standardiserer** - USB-C for AI tools
4. **Skills tilføjer viden** - uden at genopbygge

## Næste Skridt: Enterprise Agent SDK'er

Nu hvor du har bygget din egen agent-loop og forstår abstraktionerne, kan du udforske etablerede SDK'er. Disse giver dig **observability**, **multi-agent orchestration** og **enterprise compliance** ud af boksen – men bygger på præcis de samme principper, du har lært i dag.

**To tilgange dominerer:**

| | **Anthropic Claude Agent SDK** | **Microsoft Agent Framework** |
|---|---|---|
| **Filosofi** | "Giv Claude en computer" | "Unified multi-agent orchestration" |
| **Kerne** | Bash/fil-tools, subagents, MCP | AutoGen + Semantic Kernel |
| **Sprog** | Python | Python, C#, Java |
| **Styrke** | Simpel agent → computer-interaktion | Enterprise: observability, compliance, A2A |
| **MCP** | ✅ Native integration | ✅ Native integration |

![Agent SDK Landscape](images/agent-sdk-landscape.svg)

> **Nøgleindsigt:** Begge SDK'er bygger på de samme principper, du har lært (tools, loops, MCP). Forskellen ligger i *hvor meget* infrastruktur du får "gratis" – og *hvilken leverandør* du binder dig til.

## The Emperor Has No Clothes

> **"The core of these tools isn't magic. It's about 200 lines of straightforward Python."**
> — Mihail Eric

Du har nu set det selv. Du kan bygge en agent. Du forstår hvad der sker.

**Hvorfor er dette vigtigt?**

> Whilst software development/programming is now dead. We however deeply need software engineers with these skills who understand that LLMs are a new form of programmable computer. If you haven't built your own coding agent yet - please do.
> — [Geoffrey Huntley](https://ghuntley.com/loop/)

## Stof til eftertanke

Kan du komme i tanke om systemer hvor statiske formularer, der forudsætter, at brugeren på forhånd ved, hvad vedkommende har brug for, kunne erstattes af multi-modale brugergrænseflader, der forstår brugerens intention?

![Gammel vs. Ny UX-Logik](images/gammel-vs-ny-logik-ux.png)

## Ressourcer

- [MCP Documentation](https://modelcontextprotocol.io)
- [Kotlin Jupyter Kernel](https://github.com/Kotlin/kotlin-jupyter)
- [GitHub Copilot Skills](https://docs.github.com/en/copilot)
- [The Emperor Has No Clothes](https://www.mihaileric.com/The-Emperor-Has-No-Clothes/)
- [Ktor HTTP Client](https://ktor.io/docs/client-overview.html)
- [Building agents with the Claude Agent SDK](https://www.anthropic.com/engineering/building-agents-with-the-claude-agent-sdk)
- [Introducing Microsoft Agent Framework](https://azure.microsoft.com/en-us/blog/introducing-microsoft-agent-framework/)

---